In [2]:
# Import dependencies
import gc
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
import seaborn as sns
import plotly.express as px

import matplotlib.pyplot as plt
import squarify
import matplotlib.cm as cm

In [3]:
# Display settings for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)  # Display only the first 10 rows in output

In [4]:
# df_leads = None  # Clear the DataFrame
# df_opps = None  # Clear the DataFrame
# df_leads2 = None  # Clear the DataFrame
# df_leads2 = None  # Clear the DataFrame
# import gc
# gc.collect()  # Force garbage collection to remove old references

In [5]:
# Load the Salesforce report exports from Excel files
opps_3df = pd.read_excel('opportunities_export2.xlsx')  # Read default sheet for opportunities
leads_3df = pd.read_excel('leads_export2.xlsx')        # Read default sheet for leads

In [ ]:
print(leads_3df['LeadOwner'].unique())
print(opps_3df['OpportunityOwner'].unique())

In [7]:
# plt.close('all')  # This closes all active figures to clear the plotting cache

In [8]:
# Displaying column names of all dataframes
print(opps_3df.columns)
print(leads_3df.columns)

Index(['AccountName', 'OpportunityName', 'Stage', 'Type', 'CreatedDate',
       'CloseDate', 'DaysOverdue', 'AmountCurrency', 'Amount',
       'ExpectedRevenueCurrency', 'ExpectedRevenue', 'Status',
       'LastModifiedDate', 'OpportunityOwner'],
      dtype='object')
Index(['LeadStatus', 'FullName', 'CompanyName', 'LeadRecordType', 'LeadSource',
       'CreatedDate', 'CreatedBy', 'ConvertedLead', 'LastModifiedDate',
       'LeadOwner'],
      dtype='object')


In [9]:
# Ensure date columns are in datetime format
opps_3df['CloseDate'] = pd.to_datetime(opps_3df['CloseDate'])
opps_3df['CreatedDate'] = pd.to_datetime(opps_3df['CreatedDate'])
opps_3df['LastModifiedDate'] = pd.to_datetime(opps_3df['LastModifiedDate'])
leads_3df['CreatedDate'] = pd.to_datetime(leads_3df['CreatedDate'])
leads_3df['LastModifiedDate'] = pd.to_datetime(leads_3df['LastModifiedDate'])

In [10]:
# Displaying column names of all dataframes
print(opps_3df.columns)
print(leads_3df.columns)

Index(['AccountName', 'OpportunityName', 'Stage', 'Type', 'CreatedDate',
       'CloseDate', 'DaysOverdue', 'AmountCurrency', 'Amount',
       'ExpectedRevenueCurrency', 'ExpectedRevenue', 'Status',
       'LastModifiedDate', 'OpportunityOwner'],
      dtype='object')
Index(['LeadStatus', 'FullName', 'CompanyName', 'LeadRecordType', 'LeadSource',
       'CreatedDate', 'CreatedBy', 'ConvertedLead', 'LastModifiedDate',
       'LeadOwner'],
      dtype='object')


In [11]:
# describe the data types of each column, make sure showing all columns as a list
print(opps_3df.dtypes)

AccountName                        object
OpportunityName                    object
Stage                              object
Type                               object
CreatedDate                datetime64[ns]
                                ...      
ExpectedRevenueCurrency            object
ExpectedRevenue                   float64
Status                             object
LastModifiedDate           datetime64[ns]
OpportunityOwner                   object
Length: 14, dtype: object


In [12]:
# describe the data types of each column, make sure showing all columns as a list
print(leads_3df.dtypes)

LeadStatus                  object
FullName                    object
CompanyName                 object
LeadRecordType              object
LeadSource                  object
CreatedDate         datetime64[ns]
CreatedBy                   object
ConvertedLead               object
LastModifiedDate    datetime64[ns]
LeadOwner                   object
dtype: object


In [13]:
# a. Health of the Pipeline by Stage
pipeline_health = opps_3df['Stage'].value_counts().reset_index()
pipeline_health.columns = ['Stage', 'Count']

print("Pipeline Health Overview:")
print(pipeline_health)


Pipeline Health Overview:
                    Stage  Count
0              Closed Won   1471
1             Closed Lost    640
2             Prospecting    159
3              Qualifying    155
4  Evaluation/Negotiation    100
5                Proposal     80


In [ ]:
# Visualize using horizontal bar chart adding number of opportunities in each stage to the end of each bar
plt.figure(figsize=(15, 9))
sns.barplot(data=pipeline_health, y='Stage', x='Count', palette='viridis')
plt.title('Pipeline by Stage')
plt.xlabel('Number of Opportunities')
plt.ylabel('Stage')
for i in range(pipeline_health.shape[0]):
    count = pipeline_health.iloc[i]['Count']
    plt.text(count, i, count, ha='left', va='center', color='black')
plt.show()


In [ ]:
# a. Health of the Pipeline by Stage
pipeline_health = opps_3df['Stage'].value_counts().reset_index()
pipeline_health.columns = ['Stage', 'Count']

print("Pipeline Health Overview (Count by Stage):")
print(pipeline_health)

# b. Health by Stage and Opportunity Type
pipeline_health_by_type = opps_3df.groupby(['Stage', 'Type']).size().reset_index(name='Count')

print("Pipeline Health Overview by Stage and Opportunity Type:")
print(pipeline_health_by_type)

# c. Health by Stage and Opportunity Owner
pipeline_health_by_owner = opps_3df.groupby(['Stage', 'OpportunityOwner']).size().reset_index(name='Count')

print("Pipeline Health Overview by Stage and Opportunity Owner:")
print(pipeline_health_by_owner)


In [ ]:
# b. Health by Stage and Opportunity Type
pipeline_health_by_type = opps_3df.groupby(['Stage', 'Type']).size().unstack(fill_value=0)

# Plot a stacked horizontal bar chart for health by stage and opportunity type and add values to the end of each bar for each type


pipeline_health_by_type.plot(kind='barh', stacked=True, figsize=(15, 9), color=['skyblue', 'orange', 'green'])
plt.xlabel('Number of Opportunities')
plt.title('Pipeline Health by Stage and Account Type')
plt.show()



In [ ]:
# c. Health by Stage and Opportunity Owner
pipeline_health_by_owner = opps_3df.groupby(['Stage', 'OpportunityOwner']).size().unstack(fill_value=0)

# Plot a heatmap for health by stage and opportunity owner
plt.figure(figsize=(15, 9))
sns.heatmap(pipeline_health_by_owner, cmap='Blues', annot=True, fmt="d", linewidths=0.5)

plt.xlabel('Opportunity Owner')
plt.ylabel('Stage')
plt.title('Pipeline Health by Stage and Opportunity Owner (Heatmap)')
plt.show()


In [ ]:
# b. Value of the Pipeline and also add values to the end of each bar
pipeline_value = opps_3df.groupby('Stage')['Amount'].sum().reset_index()

print("Pipeline Value by Stage:")
print(pipeline_value)

print("Pipeline Value by Stage (Total):")
print(pipeline_value['Amount'].sum())

# Set up the figure
plt.figure(figsize=(15, 9))

# Create a horizontal bar chart
sns.barplot(data=pipeline_value, y='Stage', x='Amount', palette='viridis')

# Add title and labels
plt.title('Value of the Pipeline by Stage')
plt.xlabel('Amount in USD')
plt.ylabel('Stage')

# Add the value labels inside each bar
for i in range(pipeline_value.shape[0]):
    amount = pipeline_value.iloc[i]['Amount']
    plt.text(amount - (0.05 * amount), i, f"${amount:,.0f}", ha='right', va='center', color='white', fontweight='bold')

# Show the plot
plt.show()




In [19]:
# c. Overdue Opportunities
overdue_opps = opps_3df[opps_3df['CloseDate'] < datetime.now()]
print(f"Number of Overdue Opportunities: {len(overdue_opps)}")


Number of Overdue Opportunities: 2213


In [ ]:
# d. Progression by 30-Day Milestones
# Calculate the age of each opportunity in 30-day increments
opps_3df['DaysSinceCreated'] = (datetime.now() - opps_3df['CreatedDate']).dt.days
opps_3df['Milestone'] = (opps_3df['DaysSinceCreated'] // 30) + 1

# Progression of opportunities by milestone
milestone_progression = opps_3df.groupby('Milestone')['Stage'].value_counts(normalize=True).unstack(fill_value=0)

print("Progression by 30-Day Milestones:")
print(milestone_progression)

# visualize the progression of opportunities by milestone
plt.figure(figsize=(15, 9))
sns.heatmap(milestone_progression, cmap='Blues', annot=True, fmt=".1%", linewidths=0.5)

plt.xlabel('Stage')
plt.ylabel('Milestone')
plt.title('Progression of Opportunities by 30-Day Milestones (Heatmap)')
plt.show()

In [ ]:
# Calculate the age of each opportunity in 30-day increments
opps_3df['DaysSinceCreated'] = (datetime.now() - opps_3df['CreatedDate']).dt.days
opps_3df['Milestone'] = (opps_3df['DaysSinceCreated'] // 30) + 1

# Progression of opportunities by milestone (counts instead of percentages for clarity)
milestone_progression = opps_3df.groupby(['Milestone', 'Stage']).size().unstack(fill_value=0)

# Visualize the progression of opportunities by milestone with a heatmap
plt.figure(figsize=(15, 9))

# Adjust the color palette for better readability
sns.heatmap(milestone_progression, cmap='coolwarm', annot=True, fmt="d", linewidths=0.5, cbar_kws={'label': 'Number of Opportunities'})

plt.xlabel('Stage')
plt.ylabel('30-Day Milestone')
plt.title('Progression of Opportunities by 30-Day Milestones (Heatmap)')
plt.show()


In [ ]:
# Pivot the data to get counts for each stage by milestone and rep
milestone_rep_stage = opps_3df.groupby(['Milestone', 'OpportunityOwner', 'Stage']).size().unstack(fill_value=0).reset_index()

# Plot a stacked bar chart
milestone_rep_stage.groupby('Milestone').sum().plot(kind='bar', stacked=True, figsize=(15, 9), colormap='viridis')

plt.title('Progression of Opportunities by 30-Day Milestones (Stacked Bar Chart)')
plt.xlabel('30-Day Milestones')
plt.ylabel('Number of Opportunities')
plt.legend(title='Stage')
plt.show()



In [ ]:
import plotly.graph_objects as go  # Import plotly.graph_objects

# Example Milestone conversion to meaningful ranges
opps_3df['MilestoneLabel'] = pd.cut(opps_3df['Milestone'],
                                   bins=[0, 30, 60, 90, 120, 150, 180, float('inf')],
                                   labels=['0-30 days', '31-60 days', '61-90 days', '91-120 days', '121-150 days', '151-180 days', '180+ days'],
                                   right=False)

# Concatenate OpportunityOwner, Stage, and Milestone labels into a single list of labels
all_labels = pd.concat([opps_3df['OpportunityOwner'], opps_3df['Stage'], opps_3df['MilestoneLabel']]).unique()

# Create a mapping from label to index
label_to_index = {label: idx for idx, label in enumerate(all_labels)}

# Add numerical indices for OpportunityOwner, Stage, and Milestone in the dataframe
opps_3df['OwnerIndex'] = opps_3df['OpportunityOwner'].map(label_to_index)
opps_3df['StageIndex'] = opps_3df['Stage'].map(label_to_index)
opps_3df['MilestoneIndex'] = opps_3df['MilestoneLabel'].map(label_to_index)

# Group the data to get the count of opportunities for each transition (Owner → Stage → Milestone)
# 1. Owner → Stage
owner_stage_flows = opps_3df.groupby(['OwnerIndex', 'StageIndex']).size().reset_index(name='Count')

# 2. Stage → Milestone
stage_milestone_flows = opps_3df.groupby(['StageIndex', 'MilestoneIndex']).size().reset_index(name='Count')

# Combine the flows (Owner → Stage and Stage → Milestone) for visualization
all_flows = pd.concat([owner_stage_flows, stage_milestone_flows])

# Define the Sankey diagram nodes (OpportunityOwner, Stage, Milestone) and links (flows)
fig = go.Figure(go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="orange", width=0.5),
        label=all_labels,  # Use the unique owner, stage, and milestone labels
        color="purple"  # Set all node colors to purple
    ),
    link=dict(
        source=all_flows[all_flows.columns[0]],  # Source is either OwnerIndex or StageIndex
        target=all_flows[all_flows.columns[1]],  # Target is either StageIndex or MilestoneIndex
        value=all_flows['Count']  # The count of opportunities
    )
))

# Increase the height of the figure and apply layout changes
fig.update_layout(
    title_text="Flow of Opportunities from OpportunityOwner to Stages and Milestones (with Milestone Ranges)",
    font_size=15,
    height=1100  # Increase the height of the figure
)

# Display the Sankey diagram
fig.show()


In [ ]:
# Calculate the days spent in each stage by each rep
opps_3df['DaysSpent'] = (opps_3df['CloseDate'] - opps_3df['CreatedDate']).dt.days

# Group by OpportunityOwner and Stage, and calculate the mean days spent
days_spent_per_stage = opps_3df.groupby(['OpportunityOwner', 'Stage'])['DaysSpent'].mean().unstack(fill_value=0)

# Plot the heatmap
plt.figure(figsize=(15, 9))
sns.heatmap(days_spent_per_stage, cmap='coolwarm', annot=True, fmt=".1f", linewidths=0.5)

plt.title('Average Days Spent on Stage by Rep in Each Stage')
plt.xlabel('Stage')
plt.ylabel('OpportunityOwner')
plt.show()


In [ ]:
# Calculate the days spent in each stage by each rep
opps_3df['DaysSpent'] = (opps_3df['CloseDate'] - opps_3df['CreatedDate']).dt.days

# Group by OpportunityOwner and Stage, and calculate the mean days spent
days_spent_per_stage = opps_3df.groupby(['OpportunityOwner', 'Stage'])['DaysSpent'].mean().unstack(fill_value=0)

# Convert the average days spent into milestone bins
milestone_labels = pd.cut(days_spent_per_stage.values.flatten(),
                          bins=[0, 30, 60, 90, 120, 150, 180, float('inf')],
                          labels=['0-30 days', '31-60 days', '61-90 days', '91-120 days', '121-150 days', '151-180 days', '180+ days'],
                          right=False)

# Reshape the milestone labels back into the original dataframe shape
milestone_labels = milestone_labels.reshape(days_spent_per_stage.shape)

# Plot the heatmap
plt.figure(figsize=(15, 9))
sns.heatmap(days_spent_per_stage, cmap='coolwarm', annot=milestone_labels, fmt='', linewidths=0.5)  # Use fmt='' for string annotations

plt.title('Most Common 30-Day Milestone by Rep in Each Stage')
plt.xlabel('Stage')
plt.ylabel('OpportunityOwner')
plt.show()


In [ ]:
# Calculate the total number of opportunities per rep
total_opps = opps_3df.groupby('OpportunityOwner')['OpportunityName'].count()

# Calculate the number of closed opportunities per rep (assuming 'Closed Won' is the stage for successful deals)
closed_opps = opps_3df[opps_3df['Stage'] == 'Closed Won'].groupby('OpportunityOwner')['OpportunityName'].count()

# Calculate the % close rate
close_rate = (closed_opps / total_opps) * 100

# Fill any missing values with 0 (for reps with no closed opportunities)
close_rate = close_rate.fillna(0)

# Plot the % close rate as a horizontal bar chart using purple blue colors and add the % values to the end of each bar and sort

plt.figure(figsize=(15, 9))
sns.barplot(x=close_rate.values, y=close_rate.index, palette='coolwarm')
plt.title('Closed Won Rate by Rep (Key Result: This code shows the close rate for each owner—how many of their opportunities result in a won deal relative to the total opportunities they handled)')
plt.xlabel('% Close Rate')
plt.ylabel('OpportunityOwner')
for i in range(close_rate.shape[0]):
    plt.text(close_rate.values[i] + 1, i, f"{close_rate.values[i]:.1f}%", va='center')
plt.show()




In [ ]:
# analyze number of opportunities by owner
opps_by_owner = opps_3df['OpportunityOwner'].value_counts().reset_index()
opps_by_owner.columns = ['OpportunityOwner', 'Count']
opps_by_owner = opps_by_owner.sort_values('Count', ascending=False)

print("Number of Opportunities by Owner:")
print(opps_by_owner)

# visualize number of opportunities by owner and add count to the end of each bar
plt.figure(figsize=(15, 9))
sns.barplot(data=opps_by_owner, y='OpportunityOwner', x='Count', palette='viridis')

plt.title('Number of Opportunities by Owner')
plt.xlabel('Number of Opportunities')
plt.ylabel('Opportunity Owner')

for i in range(opps_by_owner.shape[0]):
    count = opps_by_owner.iloc[i]['Count']
    plt.text(count, i, count, ha='left', va='center', color='black')
    
plt.show()


In [ ]:
# identify percentages of closed won opportunities by owner and visualize adding percentage to the end of each bar
closed_won_opps = opps_3df[opps_3df['Stage'] == 'Closed Won']
closed_won_opps_by_owner = closed_won_opps['OpportunityOwner'].value_counts(normalize=True).reset_index()
closed_won_opps_by_owner.columns = ['OpportunityOwner', '% Closed Won']
closed_won_opps_by_owner = closed_won_opps_by_owner.sort_values('% Closed Won', ascending=False)

print("Percentage of Closed Won Opportunities by Owner:")
print(closed_won_opps_by_owner)

plt.figure(figsize=(15, 9))
sns.barplot(data=closed_won_opps_by_owner, y='OpportunityOwner', x='% Closed Won', palette='viridis')

plt.title('Percentage of Closed Won Opportunities by Owner (Key Result: This code shows the share of all Closed Won deals for each owner, giving insight into which owners are responsible for a larger portion of the total won deals.)')
plt.xlabel('% Closed Won')
plt.ylabel('Opportunity Owner')

for i in range(closed_won_opps_by_owner.shape[0]):
    percent = closed_won_opps_by_owner.iloc[i]['% Closed Won']
    plt.text(percent, i, f"{percent:.1%}", ha='left', va='center', color='black')

plt.show()

In [ ]:
# Filter the overdue opportunities where 'DaysOverdue' > 0 and exclude 'Closed Won' and 'Closed Lost' stages
overdue_opps_filtered = overdue_opps[(overdue_opps['DaysOverdue'] > 0) &
                                     (~overdue_opps['Stage'].isin(['Closed Won', 'Closed Lost']))]

# Count the number of overdue opportunities by owner
overdue_opps_by_owner = overdue_opps_filtered['OpportunityOwner'].value_counts().reset_index()
overdue_opps_by_owner.columns = ['OpportunityOwner', 'Count']
overdue_opps_by_owner = overdue_opps_by_owner.sort_values('Count', ascending=False)

# Print the count of overdue opportunities by owner
print("Number of Overdue Opportunities by Owner (Excluding 'Closed Won' and 'Closed Lost' with DaysOverdue > 0):")
print(overdue_opps_by_owner)

# Create the horizontal bar chart with figure size (15, 9)
plt.figure(figsize=(15, 9))
sns.barplot(data=overdue_opps_by_owner, y='OpportunityOwner', x='Count', palette='viridis')

# Set chart titles and labels
plt.title('Number of Overdue Opportunities by Owner (Excluding Closed Stages)')
plt.xlabel('Number of Overdue Opportunities')
plt.ylabel('Opportunity Owner')

# Add counts to the end of each bar
for i in range(overdue_opps_by_owner.shape[0]):
    count = overdue_opps_by_owner.iloc[i]['Count']
    plt.text(count, i, count, ha='left', va='center', color='black')

# Show the plot
plt.show()


In [ ]:
# a. Number of Leads in the System and by Rep
total_leads = leads_3df['FullName'].nunique()
leads_by_rep = leads_3df['LeadOwner'].value_counts().reset_index()
leads_by_rep.columns = ['Rep', 'Number of Leads']

print(f"Total Number of Leads: {total_leads}")
print("Number of Leads by Representative:")
print(leads_by_rep)

# visualize the number of leads by rep and add values to the end of each bar
plt.figure(figsize=(15, 9))
sns.barplot(data=leads_by_rep, y='Rep', x='Number of Leads', palette='viridis')
plt.title('Number of Leads by Representative')
plt.xlabel('Number of Leads')
plt.ylabel('Representative')
for i in range(leads_by_rep.shape[0]):
    count = leads_by_rep.iloc[i]['Number of Leads']
    plt.text(count, i, count, ha='left', va='center', color='black')
plt.show()


In [ ]:
# Filter leads that were modified within the last 30 days based on LastModifiedDate
recent_activity = leads_3df[leads_3df['LastModifiedDate'] > datetime.now() - timedelta(days=30)]

# Group by LeadOwner and count the number of leads with recent activities
recent_leads_per_rep = recent_activity.groupby('LeadOwner').size()

# Plot the recent activity in a horizontal bar chart
plt.figure(figsize=(15, 9))
recent_leads_per_rep.sort_values().plot(kind='barh', color='green')  # Sorting for better visual effect

plt.title('Leads with Recent Activities by Rep (Last 30 Days)')
plt.xlabel('Number of Leads with Recent Activity')
plt.ylabel('LeadOwner')
plt.show()




In [ ]:
# Group by LeadOwner and LeadStatus, and count the leads
leadowner_work = recent_activity.groupby(['LeadOwner', 'LeadStatus']).size().unstack(fill_value=0)

# Plot the heatmap to visualize LeadOwner and LeadStatus
plt.figure(figsize=(15, 9))
sns.heatmap(leadowner_work, cmap='coolwarm', annot=True, fmt="d", linewidths=0.5)

plt.title('Lead Owner Activity by Status (Last 30 Days)')
plt.xlabel('Lead Status')
plt.ylabel('LeadOwner')
plt.show()


In [ ]:
# Remove rows where LeadSource is blank or null
recent_activity = recent_activity[recent_activity['LeadSource'].notna() & (recent_activity['LeadSource'] != '')]

# Group by LeadStatus and LeadSource, and count the leads
leadstatus_source = recent_activity.groupby(['LeadStatus', 'LeadSource']).size().unstack(fill_value=0)

# Plot the heatmap to visualize LeadStatus and LeadSource
plt.figure(figsize=(15, 9))
sns.heatmap(leadstatus_source, cmap='coolwarm', annot=True, fmt="d", linewidths=0.5)

plt.title('Lead Status by Lead Source (Last 30 Days)')
plt.xlabel('Lead Source')
plt.ylabel('Lead Status')
plt.xticks(rotation=45, ha='right')

plt.show()



In [ ]:
# read values in LeadSource column
print(leads_3df['LeadSource'].value_counts())
